# Conformational Benchmark V2
47 protein çifti | Monomer + Multimer | 3 mod

Her protein çifti için **üç mod** çalıştırır:
1. **State1 → State2** (PDB sekans, directed)
2. **State2 → State1** (PDB sekans, directed)
3. **UniProt Seq** (State1 initial, State2 target, full UniProt sekans — sadece monomer)

**Yenilikler (V2):**
- Multi-chain desteği (chain "A,B" formatı → OF3 multimer query)
- 47 protein çifti: OC23 (25) + TP16 (12) + Additional (6) + Suggested (2) + TBD (1, skip)
- Membran proteinleri: inward/outward transporter konformasyonları
- Kategori & oligomerik state bazlı analiz

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 1: Environment Setup
# ════════════════════════════════════════════════════════════════

import os, sys, shutil, time, json, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    if os.path.exists('/content/ANM-openfold3'):
        shutil.rmtree('/content/ANM-openfold3')
    !git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

    if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
        !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
        !pip install -e /content/ANM-openfold3/openfold3-repo -q
    else:
        try:
            import openfold3
        except ImportError:
            !pip install -e /content/ANM-openfold3/openfold3-repo -q

    !pip install -q biopython matplotlib numpy torch pandas requests

    _repo_root = '/content/ANM-openfold3'
else:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

if IN_COLAB:
    of3_root = '/content/ANM-openfold3/openfold3-repo'
    if of3_root not in sys.path:
        sys.path.insert(0, of3_root)
    os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)
    from openfold3.entry_points.parameters import (
        download_model_parameters, get_default_checkpoint_dir, DEFAULT_CHECKPOINT_NAME,
    )
    _param_dir = get_default_checkpoint_dir()
    _param_dir.mkdir(parents=True, exist_ok=True)
    download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print(f'Repo root: {_repo_root}')
print('Environment ready.')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 2: Benchmark Table — 47 protein çifti (hardcoded)
# ════════════════════════════════════════════════════════════════

BENCHMARK_PAIRS = [
    {"idx": 1, "name": "Alpha-2-macroglobulin receptor-associated protein", "organism": "Bos taurus", "uniprot": "A0A075Q0W3", "pdb_s1": "6MKG", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6MKJ", "chain_s2": "A", "label_s2": "Closed", "category": "Enzyme/Binding", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 2, "name": "Uncharacterized ABC transporter", "organism": "", "uniprot": "A0A1J6PWI8", "pdb_s1": "6JN8", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6JN7", "chain_s2": "A", "label_s2": "Closed", "category": "ABC Transporter", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 3, "name": "Tyrosine kinase (EphA2)", "organism": "", "uniprot": "A0QTT2", "pdb_s1": "7CY2", "chain_s1": "A", "label_s1": "Open/Active", "pdb_s2": "7CYR", "chain_s2": "A", "label_s2": "Closed/Inactive", "category": "Kinase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 4, "name": "Histidine kinase sensor", "organism": "", "uniprot": "A2RJ53", "pdb_s1": "3FTO", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "3DRF", "chain_s2": "A", "label_s2": "Closed", "category": "Sensor kinase", "oligomeric": "Dimer", "source": "OC23"},
    {"idx": 5, "name": "Periplasmic solute-binding protein", "organism": "", "uniprot": "A6UVT1", "pdb_s1": "6HAC", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6HAV", "chain_s2": "A", "label_s2": "Closed", "category": "Binding protein", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 6, "name": "Glutamate receptor ionotropic (iGluR)", "organism": "", "uniprot": "B3EYN2", "pdb_s1": "5HO2", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "5HO0", "chain_s2": "A", "label_s2": "Closed", "category": "Ion channel", "oligomeric": "Tetramer", "source": "OC23"},
    {"idx": 7, "name": "Nucleoside triphosphate pyrophosphohydrolase", "organism": "", "uniprot": "G0S4S9", "pdb_s1": "6SL1", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6SKZ", "chain_s2": "A", "label_s2": "Closed", "category": "Hydrolase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 8, "name": "Ribose-binding protein", "organism": "", "uniprot": "G0SB58", "pdb_s1": "5MZO", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "5N2J", "chain_s2": "A", "label_s2": "Closed", "category": "Binding protein", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 9, "name": "Transferrin receptor protein 1", "organism": "", "uniprot": "O76728", "pdb_s1": "4BP8", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "4BP9", "chain_s2": "A", "label_s2": "Closed", "category": "Receptor", "oligomeric": "Homodimer", "source": "OC23"},
    {"idx": 10, "name": "Phosphoglycerate kinase 1", "organism": "", "uniprot": "P00558", "pdb_s1": "2XE6", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2YBE", "chain_s2": "A", "label_s2": "Closed", "category": "Kinase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 11, "name": "Beta-1,4-galactosyltransferase 1", "organism": "", "uniprot": "P15291", "pdb_s1": "6FWU", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2FYC", "chain_s2": "A", "label_s2": "Closed", "category": "Transferase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 12, "name": "5'-nucleotidase (CD73)", "organism": "", "uniprot": "P21589", "pdb_s1": "7QGL", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "7QGM", "chain_s2": "A", "label_s2": "Closed", "category": "Hydrolase", "oligomeric": "Homodimer", "source": "OC23"},
    {"idx": 13, "name": "Periplasmic murein peptide-binding protein (MppA)", "organism": "", "uniprot": "P31133", "pdb_s1": "6YED", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6YE8", "chain_s2": "A", "label_s2": "Closed", "category": "Binding protein", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 14, "name": "Anthrax protective antigen", "organism": "", "uniprot": "P33284", "pdb_s1": "3O80", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "3O8M", "chain_s2": "A", "label_s2": "Closed", "category": "Toxin", "oligomeric": "Heptamer", "source": "OC23"},
    {"idx": 15, "name": "Peroxisomal sarcosine oxidase", "organism": "", "uniprot": "P40131", "pdb_s1": "3TEE", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "3VKI", "chain_s2": "A", "label_s2": "Closed", "category": "Oxidoreductase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 16, "name": "Glutathione synthetase", "organism": "", "uniprot": "P48635", "pdb_s1": "2WIO", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2JJO", "chain_s2": "A", "label_s2": "Closed", "category": "Ligase", "oligomeric": "Homodimer", "source": "OC23"},
    {"idx": 17, "name": "RNA-binding protein Musashi-1", "organism": "", "uniprot": "P61316", "pdb_s1": "2ZPD", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2ZPC", "chain_s2": "A", "label_s2": "Closed", "category": "RNA-binding", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 18, "name": "eRF1 (eukaryotic release factor 1)", "organism": "", "uniprot": "P62495", "pdb_s1": "2KTV", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2KTU", "chain_s2": "A", "label_s2": "Closed", "category": "Translation factor", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 19, "name": "ABC transporter substrate-binding protein", "organism": "", "uniprot": "P71447", "pdb_s1": "6H8Z", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "5OLY", "chain_s2": "A", "label_s2": "Closed", "category": "Binding protein", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 20, "name": "Inorganic pyrophosphatase", "organism": "", "uniprot": "Q18A65", "pdb_s1": "6HNJ", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6HNI", "chain_s2": "A", "label_s2": "Closed", "category": "Hydrolase", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 21, "name": "Uncharacterized protein", "organism": "", "uniprot": "Q5F9M1", "pdb_s1": "3ZSF", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2YLN", "chain_s2": "A", "label_s2": "Closed", "category": "Unknown", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 22, "name": "Src-like adaptor protein 2 (SLAP2)", "organism": "", "uniprot": "Q5JRX3", "pdb_s1": "6XOU", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "6XOV", "chain_s2": "A", "label_s2": "Closed", "category": "Adaptor", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 23, "name": "Putative cysteine desulfurase", "organism": "", "uniprot": "Q7DAU8", "pdb_s1": "3L6G", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "3L6H", "chain_s2": "A", "label_s2": "Closed", "category": "Transferase", "oligomeric": "Homodimer", "source": "OC23"},
    {"idx": 24, "name": "mGluR1 (metabotropic glutamate receptor 1)", "organism": "", "uniprot": "Q9ERE7", "pdb_s1": "2RQM", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "2RQK", "chain_s2": "A", "label_s2": "Closed", "category": "GPCR/Receptor", "oligomeric": "Homodimer", "source": "OC23"},
    {"idx": 25, "name": "Maltose ABC transporter SBP", "organism": "", "uniprot": "Q9X6R4", "pdb_s1": "3IUM", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "3MUO", "chain_s2": "A", "label_s2": "Closed", "category": "Binding protein", "oligomeric": "Monomer", "source": "OC23"},
    {"idx": 26, "name": "ABC transporter TM287/TM288", "organism": "", "uniprot": "TM_0287", "pdb_s1": "3QF4", "chain_s1": "A,B", "label_s1": "Inward", "pdb_s2": "6QV1", "chain_s2": "A,B", "label_s2": "Outward-occluded", "category": "ABC Transporter", "oligomeric": "Heterodimer", "source": "TP16"},
    {"idx": 27, "name": "SLC1A1 (EAAT3, glutamate transporter)", "organism": "", "uniprot": "P43005", "pdb_s1": "6S3Q", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "8CUA", "chain_s2": "A", "label_s2": "Inward", "category": "SLC Transporter", "oligomeric": "Trimer", "source": "TP16"},
    {"idx": 28, "name": "SLC39 (ZIP zinc transporter)", "organism": "", "uniprot": "Q7WIR3", "pdb_s1": "6BTX", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "5AYM", "chain_s2": "A", "label_s2": "Inward", "category": "SLC Transporter", "oligomeric": "Monomer", "source": "TP16"},
    {"idx": 29, "name": "PTS glucose transporter (EIIC)", "organism": "", "uniprot": "P69786", "pdb_s1": "6BVG", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "5IWS", "chain_s2": "A", "label_s2": "Inward", "category": "PTS Transporter", "oligomeric": "Dimer", "source": "TP16"},
    {"idx": 30, "name": "MdfA (multidrug efflux pump)", "organism": "", "uniprot": "P0AEY8", "pdb_s1": "6VS1", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "6GV1", "chain_s2": "A", "label_s2": "Inward", "category": "MFS Transporter", "oligomeric": "Monomer", "source": "TP16"},
    {"idx": 31, "name": "MFSD2A (Mfsd2a transporter)", "organism": "", "uniprot": "Q8NA29", "pdb_s1": "7MJS", "chain_s1": "X", "label_s1": "Outward", "pdb_s2": "7N9A", "chain_s2": "A", "label_s2": "Inward", "category": "MFS Transporter", "oligomeric": "Monomer", "source": "TP16"},
    {"idx": 32, "name": "WlaB (ABC transporter)", "organism": "", "uniprot": "Q0P8I2", "pdb_s1": "5C78", "chain_s1": "A,B", "label_s1": "Inward", "pdb_s2": "6HRC", "chain_s2": "A,B", "label_s2": "Outward-occluded", "category": "ABC Transporter", "oligomeric": "Heterodimer", "source": "TP16"},
    {"idx": 33, "name": "Uncharacterized ABC transporter", "organism": "", "uniprot": "A5U30_003247", "pdb_s1": "6E9N", "chain_s1": "B", "label_s1": "Inward", "pdb_s2": "6E9Q", "chain_s2": "B", "label_s2": "Outward", "category": "ABC Transporter", "oligomeric": "Homodimer", "source": "TP16"},
    {"idx": 34, "name": "ABC transporter PF0708", "organism": "", "uniprot": "Q8U2B7", "pdb_s1": "6FHZ", "chain_s1": "A,B", "label_s1": "Inward", "pdb_s2": "6GWH", "chain_s2": "A,B", "label_s2": "Outward", "category": "ABC Transporter", "oligomeric": "Homodimer", "source": "TP16"},
    {"idx": 35, "name": "MurJ (lipid II flippase)", "organism": "", "uniprot": "P0AF16", "pdb_s1": "6NC7", "chain_s1": "A", "label_s1": "Inward", "pdb_s2": "6NC9", "chain_s2": "A", "label_s2": "Outward", "category": "MATE-like", "oligomeric": "Monomer", "source": "TP16"},
    {"idx": 36, "name": "AAC(3) aminoglycoside acetyltransferase", "organism": "", "uniprot": "Q4LE70", "pdb_s1": "6GCI", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "4C9J", "chain_s2": "B", "label_s2": "Inward", "category": "Acetyltransferase", "oligomeric": "Monomer", "source": "TP16"},
    {"idx": 37, "name": "ABC transporter TT_C0976", "organism": "", "uniprot": "Q72J62", "pdb_s1": "5MKK", "chain_s1": "A,B", "label_s1": "Inward", "pdb_s2": "6RAJ", "chain_s2": "A,B", "label_s2": "Outward", "category": "ABC Transporter", "oligomeric": "Homodimer", "source": "TP16"},
    {"idx": 38, "name": "P5A-ATPase (Spf1)", "organism": "", "uniprot": "Q02016", "pdb_s1": "6XMS", "chain_s1": "A", "label_s1": "E1-like", "pdb_s2": "6XMU", "chain_s2": "A", "label_s2": "E2-like", "category": "P-type ATPase", "oligomeric": "Monomer", "source": "Additional"},
    {"idx": 39, "name": "P5A-ATPase (Spf1) alt", "organism": "", "uniprot": "Q02016", "pdb_s1": "6XMQ", "chain_s1": "A", "label_s1": "E1P-like", "pdb_s2": "6XMU", "chain_s2": "A", "label_s2": "E2-like", "category": "P-type ATPase", "oligomeric": "Monomer", "source": "Additional"},
    {"idx": 40, "name": "MelB (melibiose permease)", "organism": "", "uniprot": "P0C1Y0", "pdb_s1": "7L17", "chain_s1": "A", "label_s1": "Outward", "pdb_s2": "4M64", "chain_s2": "A", "label_s2": "Inward/Partial", "category": "MFS Transporter", "oligomeric": "Monomer", "source": "Additional"},
    {"idx": 41, "name": "CymE (outer membrane transporter)", "organism": "", "uniprot": "Q07SB2", "pdb_s1": "6A6N", "chain_s1": "A", "label_s1": "Open", "pdb_s2": "7DQV", "chain_s2": "A", "label_s2": "Closed", "category": "TBDT", "oligomeric": "Monomer", "source": "Additional"},
    {"idx": 42, "name": "TmrAB (ABC transporter)", "organism": "", "uniprot": "Q72J62", "pdb_s1": "6RAF", "chain_s1": "A,B", "label_s1": "Inward", "pdb_s2": "6RAJ", "chain_s2": "A,B", "label_s2": "Outward", "category": "ABC Transporter", "oligomeric": "Heterodimer", "source": "Additional"},
    {"idx": 43, "name": "CFTR", "organism": "", "uniprot": "P13569", "pdb_s1": "5UAK", "chain_s1": "A", "label_s1": "Closed/Apo", "pdb_s2": "6MSM", "chain_s2": "A", "label_s2": "Open/Active", "category": "ABC Channel", "oligomeric": "Monomer", "source": "Additional"},
    {"idx": 44, "name": "TRPML1 (Mucolipin-1)", "organism": "", "uniprot": "Q9GZU1", "pdb_s1": "5WJ5", "chain_s1": "A", "label_s1": "Closed", "pdb_s2": "5WJ9", "chain_s2": "A", "label_s2": "Open/ML-SA1-bound", "category": "TRP Channel", "oligomeric": "Tetramer", "source": "Additional"},
    {"idx": 45, "name": "GPCR (TBD)", "organism": "", "uniprot": "-", "pdb_s1": "-", "chain_s1": "-", "label_s1": "-", "pdb_s2": "-", "chain_s2": "-", "label_s2": "-", "category": "GPCR", "oligomeric": "-", "source": "TBD"},
    {"idx": 46, "name": "SERCA (Ca2+ ATPase)", "organism": "", "uniprot": "P04191", "pdb_s1": "1SU4", "chain_s1": "A", "label_s1": "E1-Ca2+", "pdb_s2": "2C8K", "chain_s2": "A", "label_s2": "E2", "category": "P-type ATPase", "oligomeric": "Monomer", "source": "Suggested"},
    {"idx": 47, "name": "KRAS (GTPase)", "organism": "", "uniprot": "P01116", "pdb_s1": "4OBE", "chain_s1": "A", "label_s1": "Active (GTP)", "pdb_s2": "4EPR", "chain_s2": "A", "label_s2": "Inactive (GDP)", "category": "GTPase", "oligomeric": "Monomer", "source": "Suggested"},
]

# Skip entries without PDBs
SKIP_IDX = {45}  # GPCR — no PDB specified
ACTIVE_PAIRS = [p for p in BENCHMARK_PAIRS if p["idx"] not in SKIP_IDX]

# Non-standard UniProt IDs (skip mode C for these)
NON_STANDARD_UNIPROT = {"-", "TM_0287", "A5U30_003247"}

print(f'Toplam {len(BENCHMARK_PAIRS)} protein, {len(ACTIVE_PAIRS)} aktif:')
print(f'{"#":>2} {"Protein":<40} {"PDB_S1":<6} {"Chain":<5} {"PDB_S2":<6} {"Chain":<5} {"Oligo":<12} {"Source":<8}')
print('-' * 120)
for p in ACTIVE_PAIRS:
    print(f'{p["idx"]:>2} {p["name"][:40]:<40} {p["pdb_s1"]:<6} {p["chain_s1"]:<5} {p["pdb_s2"]:<6} {p["chain_s2"]:<5} {p["oligomeric"]:<12} {p["source"]:<8}')

n_mono = sum(1 for p in ACTIVE_PAIRS if "," not in p["chain_s1"])
n_multi = sum(1 for p in ACTIVE_PAIRS if "," in p["chain_s1"])
print(f'\nMonomer chains: {n_mono}, Multi-chain: {n_multi}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 3: Pipeline Configuration
# ════════════════════════════════════════════════════════════════

N_STEPS = 20

# Selective mixing (V6 best)
SELECTIVE_MIXING     = True
CHANGE_CUTOFF        = 0.15
ALPHA_BASE           = 0.05
ALPHA_MAX            = 0.7
MAPPING              = 'sigmoid'
W_CONNECTIVITY       = 0.5
W_DISTANCE           = 0.5
DIAGONAL_BAND        = 2

# Pipeline
Z_MIXING_ALPHA       = 0.30
COMBINATION_STRATEGY = 'autostop'
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# Composite gate
USE_COMPOSITE_GATE       = True
COMPOSITE_GATE_THRESHOLD = 0.55
GATE_W_CR                = 0.45
GATE_W_PLDDT             = 0.30
GATE_W_RG                = 0.15
GATE_W_PTM               = 0.10

# Hard cutoffs
CONFIDENCE_PTM_CUTOFF     = 0.15
CONFIDENCE_PLDDT_CUTOFF   = 65.0
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# Stall & drift
ALPHA_DECAY           = 0.8
MAX_STALL             = 8
ENABLE_BEST_ROLLBACK  = True
BEST_ROLLBACK_TM_DROP = 0.40
ENABLE_ADAPTIVE_STOP  = True
ADAPTIVE_STOP_WINDOW  = 3
MAX_ROLLBACK_STOP     = 5

# Fallback
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# Autostop
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# OF3
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = 5
NUM_DIFFUSION_SAMPLES = 1

# Converter
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

# Output
if IN_COLAB:
    BASE_SAVE_DIR = '/content/drive/MyDrive/ANM-openfold3/benchmark_v2'
else:
    BASE_SAVE_DIR = os.path.join(_repo_root, 'results', 'benchmark_v2')
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

# Skip mode C (UniProt) for multi-chain entries
SKIP_UNIPROT_MULTIMER = True

print(f'N_STEPS={N_STEPS}, Z_MIXING_ALPHA={Z_MIXING_ALPHA}, selective={SELECTIVE_MIXING}')
print(f'NUM_ROLLOUT_STEPS={NUM_ROLLOUT_STEPS}, NUM_DIFFUSION_SAMPLES={NUM_DIFFUSION_SAMPLES}')
print(f'SKIP_UNIPROT_MULTIMER={SKIP_UNIPROT_MULTIMER}')
print(f'Save dir: {BASE_SAVE_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 4: Helper Functions
# ════════════════════════════════════════════════════════════════

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import requests
matplotlib.rcParams['figure.dpi'] = 120

from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

# Reload src modules to pick up latest changes
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.mode_drive_utils import align_and_trim_ca
from src.of3_diffusion import load_of3_model, OF3ModelCache
from src.autostop_adapter import StructureContext


# ────────────────────────────────────────────────────────────────
#  PDB Fetch Helpers
# ────────────────────────────────────────────────────────────────

def fetch_ca(pdb_id: str, chain_id: str):
    """PDB'den CA koordinatlarini ve sekansini cek. Chain fallback var."""
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)

    chains = [c for c in structure[0].get_chains() if c.id == chain_id]
    if not chains:
        available = [c.id for c in structure[0].get_chains()]
        if available:
            print(f'    WARNING: chain {chain_id} not found in {pdb_id}, using {available[0]} (available: {available})')
            chain = [c for c in structure[0].get_chains() if c.id == available[0]][0]
        else:
            raise ValueError(f'No chains in {pdb_id}')
    else:
        chain = chains[0]

    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence


def fetch_ca_multimer(pdb_id, chain_ids_str):
    """Fetch CA coords for one or more chains, concatenated.

    Args:
        pdb_id: PDB ID
        chain_ids_str: "A" or "A,B" comma-separated

    Returns:
        coords: [N_total, 3] tensor
        sequence: concatenated sequence string
        chain_breaks: list of ints — start index of each chain segment [0, N_A, N_A+N_B]
        chain_seqs: list of per-chain sequences
    """
    chain_ids = [c.strip() for c in chain_ids_str.split(",")]
    all_coords = []
    all_seqs = []
    chain_breaks = [0]

    for cid in chain_ids:
        ca, seq = fetch_ca(pdb_id, cid)
        all_coords.append(ca)
        all_seqs.append(seq)
        chain_breaks.append(chain_breaks[-1] + len(seq))

    coords = torch.cat(all_coords, dim=0)
    sequence = "".join(all_seqs)
    return coords, sequence, chain_breaks, all_seqs


def fetch_uniprot_sequence(uniprot_id: str) -> str:
    """UniProt'tan full protein sekansini cek."""
    url = f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    lines = resp.text.strip().split('\n')
    seq = ''.join(l.strip() for l in lines if not l.startswith('>'))
    print(f'    UniProt {uniprot_id}: {len(seq)} residues')
    return seq


# ────────────────────────────────────────────────────────────────
#  UniProt Diffusion Wrapper
# ────────────────────────────────────────────────────────────────

def make_uniprot_diffusion_wrapper(diffusion_fn, zij_trunk_full, pdb_indices, n_pdb):
    """UniProt modda diffusion_fn'i wrap et.

    Pipeline z_mod [N_pdb, N_pdb, C] uretir. Bunu full UniProt boyutuna
    genisletip diffusion_fn'e ver, sonra donen CA'yi PDB pozisyonlarina indir.
    """
    idx_t = torch.tensor(pdb_indices, dtype=torch.long)
    n_uni = zij_trunk_full.shape[0]

    def _wrapped_diffusion(z_mod_pdb):
        z_full = zij_trunk_full.clone()
        z_full[idx_t.unsqueeze(1), idx_t.unsqueeze(0), :] = z_mod_pdb
        result = diffusion_fn(z_full)
        if hasattr(result, 'best_ca'):
            result.best_ca = result.best_ca[idx_t]
            result.all_ca = result.all_ca[:, idx_t, :]
            if result.plddt is not None:
                result.plddt = result.plddt[:, idx_t] if result.plddt.dim() == 2 else result.plddt[idx_t]
        else:
            result = result[idx_t]
        return result

    return _wrapped_diffusion


# ────────────────────────────────────────────────────────────────
#  Converter / Config Loaders
# ────────────────────────────────────────────────────────────────

def load_converter():
    """Converter checkpoint yukle."""
    history_path = os.path.join(DRIVE_DIR, 'history.json')
    best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

    if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        best_bf_r = -1
        best_epoch = -1
        for entry in history:
            bf_r = entry.get('val_bf_pearson', 0)
            if bf_r > best_bf_r:
                best_bf_r = bf_r
                best_epoch = entry['epoch']
        epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
        ckpt_path = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
        print(f'  Converter: best bf_r={best_bf_r:.4f} epoch={best_epoch}')
    else:
        ckpt_path = best_model_path
    return PairContactConverter(ckpt_path)


def make_config(chain_id):
    """Pipeline config olustur."""
    return ModeDriveConfig(
        n_steps=N_STEPS,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=Z_MIXING_ALPHA,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        use_composite_gate=USE_COMPOSITE_GATE,
        composite_gate_threshold=COMPOSITE_GATE_THRESHOLD,
        gate_w_cr=GATE_W_CR,
        gate_w_plddt=GATE_W_PLDDT,
        gate_w_rg=GATE_W_RG,
        gate_w_ptm=GATE_W_PTM,
        confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
        confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=chain_id,
        rejected_alpha_decay=ALPHA_DECAY,
        max_consecutive_rejected=MAX_STALL,
        confidence_warmup_steps=0,
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
        selective_mixing=SELECTIVE_MIXING,
        selective_w_connectivity=W_CONNECTIVITY,
        selective_w_distance=W_DISTANCE,
        selective_change_cutoff=CHANGE_CUTOFF,
        selective_alpha_base=ALPHA_BASE,
        selective_alpha_max=ALPHA_MAX,
        selective_mapping=MAPPING,
        selective_diagonal_band=DIAGONAL_BAND,
    )


# ────────────────────────────────────────────────────────────────
#  Main Pipeline Runner
# ────────────────────────────────────────────────────────────────

def run_single_direction(converter, of3_cache, pair, direction='s1_to_s2',
                         mode='directed', uniprot_seq=None):
    """Run pipeline for one direction.

    Args:
        pair: dict from BENCHMARK_PAIRS
        direction: 's1_to_s2' or 's2_to_s1'
        mode: 'directed' or 'uniprot'
        uniprot_seq: full UniProt sequence (for mode='uniprot')

    Returns:
        result dict with TM/RMSD metrics, or None on failure
    """
    if direction == 's1_to_s2':
        init_pdb, init_chain = pair["pdb_s1"], pair["chain_s1"]
        tgt_pdb, tgt_chain = pair["pdb_s2"], pair["chain_s2"]
        label = f'{pair["label_s1"]} -> {pair["label_s2"]}'
    else:
        init_pdb, init_chain = pair["pdb_s2"], pair["chain_s2"]
        tgt_pdb, tgt_chain = pair["pdb_s1"], pair["chain_s1"]
        label = f'{pair["label_s2"]} -> {pair["label_s1"]}'

    is_multichain = "," in init_chain

    save_dir = os.path.join(BASE_SAVE_DIR, f'{pair["idx"]:02d}_{pair["name"][:20].replace(" ", "_")}')
    os.makedirs(save_dir, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── Fetch structures ──
    if is_multichain:
        initial_ca, seq_initial, chain_breaks_init, chain_seqs_init = fetch_ca_multimer(init_pdb, init_chain)
        target_ca_raw, seq_target, chain_breaks_tgt, chain_seqs_tgt = fetch_ca_multimer(tgt_pdb, tgt_chain)
    else:
        initial_ca, seq_initial = fetch_ca(init_pdb, init_chain)
        target_ca_raw, seq_target = fetch_ca(tgt_pdb, tgt_chain)
        chain_seqs_init = [seq_initial]

    N_init = initial_ca.shape[0]
    N_target = target_ca_raw.shape[0]

    # ── Size mismatch handling ──
    if N_init != N_target:
        print(f'    SIZE MISMATCH: initial N={N_init}, target N={N_target} -> alignment')
        ca_init_trim, ca_tgt_trim, common_seq, idx_init, idx_target = align_and_trim_ca(
            initial_ca, seq_initial, target_ca_raw, seq_target
        )
        print(f'    Common core: {len(common_seq)} residues')
        target_ca = ca_tgt_trim
        initial_ca_for_comparison = ca_init_trim
        comparison_indices = idx_init
    else:
        target_ca = target_ca_raw
        initial_ca_for_comparison = initial_ca
        comparison_indices = None

    init_tm = tm_score(initial_ca_for_comparison, target_ca)
    init_rmsd = compute_rmsd(initial_ca_for_comparison, target_ca)
    print(f'    N={N_init}, init TM={init_tm:.4f}, init RMSD={init_rmsd:.2f}A ({label})')

    # ── OF3 sequence selection & diffusion_fn ──
    if mode == 'uniprot' and uniprot_seq is not None:
        of3_sequence = uniprot_seq
        N_of3 = len(of3_sequence)
        print(f'    OF3 sequence: UniProt ({N_of3} res) vs PDB ({N_init} res)')

        diffusion_fn_full, zij_trunk_full = of3_cache.prepare_sequence(
            sequence=of3_sequence,
            query_name=init_pdb,
            chain_id=init_chain.split(",")[0] if is_multichain else init_chain,
            num_rollout_steps=NUM_ROLLOUT_STEPS,
        )

        if N_of3 != N_init:
            from Bio.Align import PairwiseAligner
            _aligner = PairwiseAligner()
            _aligner.mode = 'global'
            _aligner.match_score = 2
            _aligner.mismatch_score = -1
            _aligner.open_gap_score = -5
            _aligner.extend_gap_score = -0.5
            _aln = _aligner.align(of3_sequence, seq_initial)[0]

            pdb_in_uniprot = []
            pos_uni, pos_pdb = 0, 0
            for char_u, char_p in zip(str(_aln[0]), str(_aln[1])):
                gap_u = (char_u == '-')
                gap_p = (char_p == '-')
                if not gap_u and not gap_p:
                    pdb_in_uniprot.append(pos_uni)
                if not gap_u:
                    pos_uni += 1
                if not gap_p:
                    pos_pdb += 1

            print(f'    PDB->UniProt mapping: {len(pdb_in_uniprot)} positions matched')

            idx_t = torch.tensor(pdb_in_uniprot, dtype=torch.long)
            zij_trunk = zij_trunk_full[idx_t][:, idx_t, :]
            print(f'    zij_trunk trimmed: {zij_trunk.shape}')

            diffusion_fn = make_uniprot_diffusion_wrapper(
                diffusion_fn_full, zij_trunk_full, pdb_in_uniprot, N_init
            )
        else:
            zij_trunk = zij_trunk_full
            diffusion_fn = diffusion_fn_full
    elif is_multichain:
        # Multi-chain OF3 query
        chains_data = [
            {"chain_id": cid.strip(), "sequence": seq}
            for cid, seq in zip(init_chain.split(","), chain_seqs_init)
        ]
        diffusion_fn, zij_trunk = of3_cache.prepare_sequence(
            sequence=seq_initial,  # ignored when chains_data is set
            query_name=init_pdb,
            chain_id=init_chain.split(",")[0],
            num_rollout_steps=NUM_ROLLOUT_STEPS,
            chains_data=chains_data,
        )
    else:
        # Single-chain directed mode
        diffusion_fn, zij_trunk = of3_cache.prepare_sequence(
            sequence=seq_initial,
            query_name=init_pdb,
            chain_id=init_chain,
            num_rollout_steps=NUM_ROLLOUT_STEPS,
        )

    # ── Structure context (for autostop) ──
    _pdb_file = PDBList().retrieve_pdb_file(init_pdb, pdir='/tmp/pdb_cache', file_format='pdb')
    _autostop_chain = init_chain.split(",")[0] if is_multichain else init_chain
    structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=_autostop_chain)

    # ── Config & Pipeline ──
    cfg = make_config(_autostop_chain)
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    # ── Step loop ──
    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = Z_MIXING_ALPHA
    current_alpha = orig_alpha
    consecutive_rejected = 0
    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    best_target_tm = init_tm
    rollback_count = 0

    trajectory_coords = [initial_ca.clone()]
    step_tms_target = [init_tm]
    step_rmsds_target = [init_rmsd]
    step_tms_init = []
    step_rmsds_init = []
    first_predicted_ca = None
    step_metrics = []

    t0 = time.time()

    for step_idx in range(N_STEPS):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca if comparison_indices is None else None,
            step_idx=step_idx,
        )

        accepted = not sr.rejected
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            accepted = False
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            accepted = False
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            accepted = False

        # Trim predicted CA for comparison
        predicted_ca_cmp = sr.new_ca[comparison_indices] if comparison_indices is not None else sr.new_ca

        cur_tm = tm_score(predicted_ca_cmp, target_ca)
        cur_rmsd = compute_rmsd(predicted_ca_cmp, target_ca)

        # First predict baseline
        if first_predicted_ca is None and accepted:
            first_predicted_ca = predicted_ca_cmp.clone()

        if first_predicted_ca is not None:
            tm_vs_first = tm_score(predicted_ca_cmp, first_predicted_ca)
            rmsd_vs_first = compute_rmsd(predicted_ca_cmp, first_predicted_ca)
        else:
            tm_vs_first = 1.0
            rmsd_vs_first = 0.0

        plddt_mean = float(sr.plddt.mean()) if sr.plddt is not None else None

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'tm_to_target': cur_tm,
            'rmsd_to_target': cur_rmsd,
            'tm_to_first': tm_vs_first,
            'rmsd_to_first': rmsd_vs_first,
            'rmsd_to_init': sr.rmsd,
            'ptm': sr.ptm,
            'plddt_mean': plddt_mean,
            'contact_recon': sr.contact_recon,
            'rg_ratio': sr.rg_ratio,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
        }
        step_metrics.append(m)

        tag = 'ACC' if accepted else 'REJ'
        print(
            f'      [{step_idx+1:>2}/{N_STEPS}] {tag} '
            f'TM_tgt={cur_tm:.3f} RMSD_tgt={cur_rmsd:.1f} '
            f'TM_1st={tm_vs_first:.3f} FL={sr.fallback_level}'
        )

        if accepted:
            trajectory_coords.append(sr.new_ca.clone())
            step_tms_target.append(cur_tm)
            step_rmsds_target.append(cur_rmsd)
            step_tms_init.append(tm_vs_first)
            step_rmsds_init.append(rmsd_vs_first)
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            # Best tracking
            if cur_tm > best_target_tm:
                best_target_tm = cur_tm
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
            elif best_target_tm > 0 and cur_tm < best_target_tm * (1.0 - BEST_ROLLBACK_TM_DROP):
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                if rollback_count >= MAX_ROLLBACK_STOP:
                    print(f'      STOP: {rollback_count} rollback')
                    break
        else:
            consecutive_rejected += 1
            if ALPHA_DECAY < 1.0:
                current_alpha = max(0.02, current_alpha * ALPHA_DECAY)
            if MAX_STALL > 0 and consecutive_rejected >= MAX_STALL:
                print(f'      STALL: {consecutive_rejected} rejected')
                break

    wall = time.time() - t0

    # ── Result ──
    final_tm = step_tms_target[-1]
    final_rmsd = step_rmsds_target[-1]
    n_accepted = len(trajectory_coords) - 1

    result = {
        'init_tm': init_tm,
        'init_rmsd': init_rmsd,
        'best_tm': best_target_tm,
        'final_tm': final_tm,
        'final_rmsd': final_rmsd,
        'delta_tm': best_target_tm - init_tm,
        'n_accepted': n_accepted,
        'n_steps_run': len(step_metrics),
        'rollbacks': rollback_count,
        'wall_time': wall,
        'step_tms_target': step_tms_target,
        'step_rmsds_target': step_rmsds_target,
        'step_tms_init': step_tms_init,
        'step_rmsds_init': step_rmsds_init,
        'step_metrics': step_metrics,
        'trajectory_coords': trajectory_coords,
        'mode': mode,
    }

    print(f'    => init_TM={init_tm:.4f} best_TM={best_target_tm:.4f} delta={best_target_tm-init_tm:+.4f} ({wall:.0f}s)')
    return result


print('Helper fonksiyonlar tanimlandi.')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 5: Load Converter + OF3 Model (bir kez)
# ════════════════════════════════════════════════════════════════

converter = load_converter()
print('Converter ready.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
of3_cache = load_of3_model(
    device=device,
    num_samples=NUM_DIFFUSION_SAMPLES,
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
)
print('OF3 model cached — her protein icin prepare_sequence() kullanilacak.')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 6: Main Benchmark Loop
# ════════════════════════════════════════════════════════════════

all_results = []

for pair_idx, pair in enumerate(ACTIVE_PAIRS):
    idx = pair["idx"]
    name = pair["name"]
    oligomeric = pair["oligomeric"]
    is_multichain = "," in pair["chain_s1"]

    print(f'\n{"="*80}')
    print(f'  [{pair_idx+1}/{len(ACTIVE_PAIRS)}] {name} ({oligomeric})')
    print(f'  {pair["pdb_s1"]}({pair["label_s1"]}) <-> {pair["pdb_s2"]}({pair["label_s2"]})')
    if is_multichain:
        print(f'  Multi-chain: {pair["chain_s1"]}')
    print(f'{"="*80}')

    # ── [A] State1 -> State2 ──
    print(f'\n  [A] {pair["pdb_s1"]}({pair["label_s1"]}) -> {pair["pdb_s2"]}({pair["label_s2"]})')
    try:
        res_a = run_single_direction(converter, of3_cache, pair, 's1_to_s2', mode='directed')
        if res_a is not None:
            all_results.append({
                **pair,
                "direction": f'{pair["pdb_s1"]}->{pair["pdb_s2"]}',
                "dir_type": "s1->s2",
                "result": res_a,
            })
    except Exception as e:
        print(f'    HATA: {e}')
        import traceback; traceback.print_exc()

    # ── [B] State2 -> State1 ──
    print(f'\n  [B] {pair["pdb_s2"]}({pair["label_s2"]}) -> {pair["pdb_s1"]}({pair["label_s1"]})')
    try:
        res_b = run_single_direction(converter, of3_cache, pair, 's2_to_s1', mode='directed')
        if res_b is not None:
            all_results.append({
                **pair,
                "direction": f'{pair["pdb_s2"]}->{pair["pdb_s1"]}',
                "dir_type": "s2->s1",
                "result": res_b,
            })
    except Exception as e:
        print(f'    HATA: {e}')
        import traceback; traceback.print_exc()

    # ── [C] UniProt — skip for multi-chain or non-standard UniProt ──
    skip_uniprot = (
        (is_multichain and SKIP_UNIPROT_MULTIMER)
        or pair["uniprot"] in NON_STANDARD_UNIPROT
    )
    if not skip_uniprot:
        print(f'\n  [C] UniProt {pair["uniprot"]} ({pair["pdb_s1"]} -> {pair["pdb_s2"]})')
        try:
            uni_seq = fetch_uniprot_sequence(pair["uniprot"])
            res_c = run_single_direction(
                converter, of3_cache, pair, 's1_to_s2',
                mode='uniprot', uniprot_seq=uni_seq,
            )
            if res_c is not None:
                all_results.append({
                    **pair,
                    "direction": f'{pair["uniprot"]}(uni)',
                    "dir_type": "uniprot",
                    "result": res_c,
                })
        except Exception as e:
            print(f'    HATA UniProt: {e}')
            import traceback; traceback.print_exc()
    else:
        reason = "multi-chain" if is_multichain else "non-standard UniProt"
        print(f'\n  [C] SKIP UniProt ({reason})')

print(f'\n\n{"="*80}')
print(f'BENCHMARK TAMAMLANDI: {len(all_results)} basarili run / {len(ACTIVE_PAIRS)} protein')
print(f'{"="*80}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 7: Summary Table + CSV
# ════════════════════════════════════════════════════════════════

summary_rows = []
for entry in all_results:
    res = entry["result"]
    row = {
        'Protein': entry["name"][:30],
        'Direction': entry["direction"],
        'Mode': entry["dir_type"],
        'Oligomeric': entry["oligomeric"],
        'Category': entry["category"],
        'Source': entry["source"],
        'Init TM': f"{res['init_tm']:.4f}",
        'Best TM': f"{res['best_tm']:.4f}",
        'Delta TM': f"{res['delta_tm']:+.4f}",
        'Init RMSD': f"{res['init_rmsd']:.2f}",
        'Final RMSD': f"{res['final_rmsd']:.2f}",
        'Accepted': f"{res['n_accepted']}/{res['n_steps_run']}",
        'Time(s)': f"{res['wall_time']:.0f}",
    }
    summary_rows.append(row)

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

csv_path = os.path.join(BASE_SAVE_DIR, 'benchmark_v2_summary.csv')
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Mode bazinda ortalamalar
for mode_name in ['s1->s2', 's2->s1', 'uniprot']:
    mode_results = [e for e in all_results if e["dir_type"] == mode_name]
    if mode_results:
        avg_d = np.mean([e["result"]["delta_tm"] for e in mode_results])
        avg_b = np.mean([e["result"]["best_tm"] for e in mode_results])
        pos = sum(1 for e in mode_results if e["result"]["delta_tm"] > 0.01)
        print(f'\n[{mode_name}] n={len(mode_results)}, avg_delta={avg_d:+.4f}, avg_best={avg_b:.4f}, pozitif={pos}/{len(mode_results)}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 8: Visualizations — 4 plots
# ════════════════════════════════════════════════════════════════

if all_results:
    source_colors = {'OC23': '#3498db', 'TP16': '#e74c3c', 'Additional': '#27ae60', 'Suggested': '#9b59b6'}

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    # ── Plot 1: Delta TM bar chart colored by source ──
    ax = axes[0, 0]
    labels = [f'{e["name"][:18]}\n{e["direction"]}' for e in all_results]
    deltas = [e["result"]["delta_tm"] for e in all_results]
    colors = [source_colors.get(e["source"], '#95a5a6') for e in all_results]

    y_pos = np.arange(len(labels))
    # Show at most 40 bars to keep readable; if more, use smaller font
    if len(labels) > 40:
        ax.barh(y_pos, deltas, color=colors, edgecolor='none', alpha=0.8, height=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels, fontsize=4)
    else:
        ax.barh(y_pos, deltas, color=colors, edgecolor='black', linewidth=0.3, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels, fontsize=5)

    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Delta TM (best - initial)')
    ax.set_title('Delta TM by Source')
    ax.grid(True, alpha=0.2, axis='x')

    from matplotlib.patches import Patch
    legend_patches = [Patch(facecolor=c, label=s) for s, c in source_colors.items()]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

    # ── Plot 2: Scatter — Init TM vs Best TM (monomer vs multimer markers) ──
    ax = axes[0, 1]
    for e in all_results:
        res = e["result"]
        is_multi = "," in e["chain_s1"]
        marker = 's' if is_multi else 'o'
        color = source_colors.get(e["source"], '#95a5a6')
        ax.scatter(res['init_tm'], res['best_tm'], c=color, marker=marker,
                  s=80, edgecolors='black', linewidths=0.5, alpha=0.8, zorder=5)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='No change')
    ax.set_xlabel('Initial TM to Target')
    ax.set_ylabel('Best TM to Target')
    ax.set_title('Init vs Best TM\n(circle=monomer, square=multimer)')
    ax.set_xlim(0.2, 1.0)
    ax.set_ylim(0.2, 1.0)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

    from matplotlib.lines import Line2D
    legend_el = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='Monomer'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=8, label='Multimer'),
    ]
    ax.legend(handles=legend_el, loc='lower right', fontsize=8)

    # ── Plot 3: Box plot — Delta TM by category ──
    ax = axes[1, 0]
    from collections import defaultdict
    cat_deltas = defaultdict(list)
    for e in all_results:
        cat_deltas[e["category"]].append(e["result"]["delta_tm"])

    # Sort by median delta TM
    sorted_cats = sorted(cat_deltas.keys(), key=lambda c: np.median(cat_deltas[c]), reverse=True)
    box_data = [cat_deltas[c] for c in sorted_cats]

    bp = ax.boxplot(box_data, vert=False, patch_artist=True, labels=sorted_cats)
    for patch in bp['boxes']:
        patch.set_facecolor('#ecf0f1')
        patch.set_edgecolor('#2c3e50')
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Delta TM')
    ax.set_title('Delta TM by Category')
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(True, alpha=0.2, axis='x')

    # ── Plot 4: Success rate by oligomeric state ──
    ax = axes[1, 1]
    oligo_stats = defaultdict(lambda: {'total': 0, 'success': 0})
    for e in all_results:
        oligo = e["oligomeric"]
        oligo_stats[oligo]['total'] += 1
        if e["result"]["delta_tm"] > 0.01:
            oligo_stats[oligo]['success'] += 1

    oligo_names = sorted(oligo_stats.keys())
    success_rates = [oligo_stats[o]['success'] / oligo_stats[o]['total'] * 100 for o in oligo_names]
    totals = [oligo_stats[o]['total'] for o in oligo_names]

    bars = ax.bar(oligo_names, success_rates, color='#2ecc71', edgecolor='black', linewidth=0.5, alpha=0.8)
    for bar, total, rate in zip(bars, totals, success_rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'n={total}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Success Rate (%)')
    ax.set_xlabel('Oligomeric State')
    ax.set_title('Success Rate by Oligomeric State\n(Delta TM > 0.01 = success)')
    ax.set_ylim(0, 110)
    ax.grid(True, alpha=0.2, axis='y')
    ax.tick_params(axis='x', rotation=30, labelsize=8)

    plt.suptitle('Conformational Benchmark V2 — Overview', fontsize=14, y=1.01)
    plt.tight_layout()
    fig_path = os.path.join(BASE_SAVE_DIR, 'fig_benchmark_v2_overview.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
else:
    print('No results to plot.')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 9: Per-source Statistics
# ════════════════════════════════════════════════════════════════

print('='*80)
print('  PER-SOURCE STATISTICS')
print('='*80)

for source in ['OC23', 'TP16', 'Additional', 'Suggested']:
    source_results = [e for e in all_results if e["source"] == source]
    if source_results:
        deltas = [e["result"]["delta_tm"] for e in source_results]
        avg_delta = np.mean(deltas)
        med_delta = np.median(deltas)
        n_pos = sum(1 for d in deltas if d > 0.01)
        n_neg = sum(1 for d in deltas if d < -0.01)
        avg_time = np.mean([e["result"]["wall_time"] for e in source_results])
        print(f'\n[{source}] n={len(source_results)}')
        print(f'  avg_delta={avg_delta:+.4f}  median_delta={med_delta:+.4f}')
        print(f'  positive={n_pos}/{len(source_results)}  negative={n_neg}/{len(source_results)}')
        print(f'  avg_time={avg_time:.0f}s')

# Monomer vs Multimer comparison
print(f'\n{"─"*40}')
mono_results = [e for e in all_results if "," not in e["chain_s1"]]
multi_results = [e for e in all_results if "," in e["chain_s1"]]

if mono_results:
    avg_d = np.mean([e["result"]["delta_tm"] for e in mono_results])
    n_pos = sum(1 for e in mono_results if e["result"]["delta_tm"] > 0.01)
    print(f'\nMonomer: n={len(mono_results)}, avg_delta={avg_d:+.4f}, positive={n_pos}/{len(mono_results)}')

if multi_results:
    avg_d = np.mean([e["result"]["delta_tm"] for e in multi_results])
    n_pos = sum(1 for e in multi_results if e["result"]["delta_tm"] > 0.01)
    print(f'Multimer: n={len(multi_results)}, avg_delta={avg_d:+.4f}, positive={n_pos}/{len(multi_results)}')

# Overall
if all_results:
    total_time = sum(e["result"]["wall_time"] for e in all_results)
    n_total = len(all_results)
    n_positive = sum(1 for e in all_results if e["result"]["delta_tm"] > 0.01)
    avg_delta_all = np.mean([e["result"]["delta_tm"] for e in all_results])
    print(f'\n{"─"*40}')
    print(f'OVERALL: n={n_total}, avg_delta={avg_delta_all:+.4f}, positive={n_positive}/{n_total} ({100*n_positive/n_total:.0f}%)')
    print(f'Total time: {total_time:.0f}s ({total_time/60:.1f}min)')

print('='*80)